# Topics & Partitions: Ordering and Distribution

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

**This topic runs no Spark job.** Every cell below talks to the brokers directly over the network (`kafka-1:9092` / `kafka-2:9092` / `kafka-3:9092`, reachable in-cluster) using `kafka-python`, exactly like `kafka-architecture-kraft`'s one Python step -- no Docker CLI/socket access is needed for anything in this notebook.

In [ ]:
from collections import Counter

from kafka import KafkaConsumer, KafkaProducer, TopicPartition

BOOTSTRAP_SERVERS = ["kafka-1:9092", "kafka-2:9092", "kafka-3:9092"]
TOPIC = "topics-partitions-demo"

producer = KafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
print("Connected to the cluster:", producer.bootstrap_connected())

## 1. Same key, same partition -- every time

The broker auto-creates `topics-partitions-demo` with 3 partitions on first send (`KAFKA_NUM_PARTITIONS=3` on this cluster, same auto-create mechanism `kafka-architecture-kraft` used). `kafka-python`'s producer hashes the key (murmur2, matching the Java client) to pick a partition -- the same key bytes always hash to the same partition for a given partition count, so every message under one key lands together and stays ordered relative to each other (see `concept.md`).

Below, three distinct keys (`user-a`, `user-c`, `user-8`) each send 4 messages. Watch the printed partition per send: it should never change within a key.

In [ ]:
KEYS = ["user-a", "user-c", "user-8"]
key_partitions = {}
key_first_offsets = {}

for key in KEYS:
    for seq in range(4):
        future = producer.send(TOPIC, key=key.encode(), value=f"{key}-event-{seq}".encode())
        metadata = future.get(timeout=10)
        key_partitions.setdefault(key, set()).add(metadata.partition)
        key_first_offsets.setdefault(key, metadata.offset)
        print(f"key={key!r:10} seq={seq} -> partition {metadata.partition}, offset {metadata.offset}")

producer.flush()
print()
for key, partitions in key_partitions.items():
    print(f"{key}: all 4 sends landed on partition(s) {partitions}")
    assert len(partitions) == 1, f"{key} spread across more than one partition -- unexpected"


## 2. No key -- how `kafka-python` actually spreads messages

With no key, the producer has no hash input to pin a partition, so it must pick one some other way. Two behaviors show up across Kafka client versions/languages: classic **round-robin** (partition 0, 1, 2, 0, 1, 2, ...) or the newer **sticky partitioner** (batch a run of messages onto one partition at a time, then switch, to build bigger batches). This repo pins `kafka-python==2.0.2` -- rather than assume, send 30 unkeyed messages and look at the raw per-send partition sequence.

In [ ]:
NO_KEY_SEND_COUNT = 30
no_key_partitions = []

for seq in range(NO_KEY_SEND_COUNT):
    future = producer.send(TOPIC, value=f"no-key-event-{seq}".encode())
    metadata = future.get(timeout=10)
    no_key_partitions.append(metadata.partition)

producer.flush()
print("partition sequence:", no_key_partitions)
print("counts per partition:", dict(Counter(no_key_partitions)))

## 3. Naming the observed behavior

Read the partition sequence printed above. If it were sticky-partitioner batching, you'd see long runs of the same partition repeated (e.g. `0,0,0,0,0,1,1,1,...`) before switching. If it were classic round-robin, you'd see a strict repeating cycle (`0,1,2,0,1,2,...`).

`kafka-python==2.0.2`'s `DefaultPartitioner` (`kafka/partitioner/default.py`) does neither: when `key is None` it calls `random.choice(available)` **per message** -- an independent uniformly-random partition choice on every single send, not a batch-sticky run and not a fixed cycle. `concept.md` names this explicitly. Over enough sends the counts-per-partition above should still land roughly even (random, not skewed), but the *sequence* itself has no run structure or cycle -- that's the observable signature of per-message random selection rather than either textbook strategy.

## 4. Per-partition ordering holds; there is no cross-partition guarantee

Read back `user-a`'s 4 messages from step 1 by consuming **only the partition they all landed on** (`TopicPartition(TOPIC, partition)`), seeking to the start. Kafka guarantees order *within* a partition -- the 4 `user-a-event-N` messages should come back in the exact `seq=0,1,2,3` order they were produced in.

In [ ]:
target_partition = next(iter(key_partitions["user-a"]))
# Read back exactly the 4 messages just produced for this key, starting at
# the exact offset they were assigned (robust to a re-run adding more data
# to this same demo topic -- reading "from the beginning" would otherwise
# also pick up an earlier run's messages).
start_offset = key_first_offsets["user-a"]

consumer = KafkaConsumer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    consumer_timeout_ms=5000,
)
tp = TopicPartition(TOPIC, target_partition)
consumer.assign([tp])
consumer.seek(tp, start_offset)

read_back = []
for msg in consumer:
    read_back.append(msg.value.decode())
    if len(read_back) == 4:
        break
consumer.close()

print(f"partition {target_partition}, user-a messages in read order:", read_back)
expected = [f"user-a-event-{seq}" for seq in range(4)]
assert read_back == expected, "per-partition order was NOT preserved -- unexpected"
print("Per-partition order preserved: matches produce order exactly.")


## 5. What this does *not* prove

Step 4 only reads **one** partition, and only that partition's order is guaranteed. Nothing above establishes any ordering *between* partitions -- step 2's no-key sends (`no-key-event-0`, `no-key-event-1`, ...) landed on whichever of the 3 partitions the random partitioner picked each time, so reading partitions 0, 1, and 2 back independently and interleaving them by wall-clock/offset would **not** reproduce the original `0, 1, 2, ...` send order. This is the concept a learner most often gets wrong: Kafka orders *within* a partition, never *across* partitions of the same topic. A consumer that needs global order across all of a topic's traffic has to route every message that must stay ordered relative to each other through the *same* partition -- i.e. the same key, exactly what step 1 demonstrated.